<a href="https://colab.research.google.com/github/SZ2012-8CYL1ND3R/symmetrical-system/blob/main/Agentic_AI_Day_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests

def get_detailed_weather(city, unit="Celsius"):
    # Step 1: Call the Geocoding API to resolve any city name into coordinates
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"

    try:
        geo_response = requests.get(geo_url, timeout=5)

        if geo_response.status_code != 200:
            return f"Error: Geocoding server returned status code {geo_response.status_code}."

        geo_data = geo_response.json()

        # Check if the API found a matching city
        if "results" not in geo_data or not geo_data["results"]:
            return f"Sorry Salman, I couldn't find geographic coordinates for '{city}' anywhere on Earth!"

        # Extract latitude and longitude from the geocoding response
        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        resolved_name = geo_data["results"][0]["name"]

        # Step 2: Chain those coordinates into the Weather Forecast API
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m"

        weather_response = requests.get(weather_url, timeout=5)

        if weather_response.status_code != 200:
            return f"Error: Weather server returned status code {weather_response.status_code}."

        weather_data = weather_response.json()
        temp = weather_data["current"]["temperature_2m"]

        if unit.lower() == "fahrenheit":
            temp = (temp * 9/5) + 32

        return f"The live temperature in {resolved_name} is {temp}° {unit}."

    except requests.exceptions.RequestException as e:
        return f"Network Error: Unable to connect to mapping or weather services. ({e})"

def control_car_climate(car, temperature):
  return f"Setting the climate control in your {car} to {temperature}°C."

tool_dispatcher = {
    "detailed_weather": get_detailed_weather,
    "car_climate": control_car_climate
}

conversation_history = [
    {"role": "system", "content": "You are Nova, an advanced, autonomous agent with live API capabilities."}
]

print("🤖 Nova Agent Initialized with Live API Integration.")
print("Type 'exit' or 'quit' at any time to end the session.\n")

while True:
  user_input = input("USER: ")

  if user_input.lower() in ["exit", "quit"]:
    print("ASSISTANT: Goodbye, Salman. Have a wonderful day.")
    break

  conversation_history.append({"role": "user", "content": user_input})

  if len(conversation_history) > 7:
    system_prompt = conversation_history[0]
    recent_messages = conversation_history[-6:]
    conversation_history = [system_prompt] + recent_messages
    print("⚙️ [System: Memory trimmed to keep agent fast.]")

  lower_input = user_input.lower()
  words = user_input.split()

  triggered_tools = []
  tool_results = []

  if "weather" in lower_input:
    city = "Toronto"
    unit = "Celsius"

    # Check for unit preference
    if "fahrenheit" in lower_input:
      unit = "Fahrenheit"

    for preposition in ["in", "at"]:
      if preposition in [w.lower() for w in words]:
        lower_words = [w.lower() for w in words]
        idx = lower_words.index(preposition)
        if idx + 1 < len(words):
          city = words[idx + 1].strip("?.,!")

    # Package into a structured dictionary with arguments
    triggered_tools.append({
        "tool": "detailed_weather",
        "args": {"city": city, "unit": unit}
    })

  if "climate" in lower_input or "ac" in lower_input:
    triggered_tools.append({
        "tool": "car_climate",
        "args": {"car": "Mercedes", "temperature": 21}
    })

  if not triggered_tools:
    fallback_message = "I'm sorry, I couldn't figure out which tool to use. Try asking about the live weather or car climate."
    conversation_history.append({"role": "assistant", "content": fallback_message})
    print(f"ASSISTANT: {fallback_message}\n")
    continue

  for item in triggered_tools:
    t_name = item["tool"]
    t_args = item["args"] # The dictionary package

    if t_name in tool_dispatcher:
      func = tool_dispatcher[t_name]
      # Python's ** unpacks the dictionary keys directly into function parameters
      result = func(**t_args)
      tool_results.append(result)
    else:
      tool_results.append(f"Error: tool {t_name} not found.")

  final_output = "  |  ".join(tool_results)
  conversation_history.append({"role": "assistant", "content": final_output})
  print(f"ASSISTANT: {final_output}\n")

🤖 Nova Agent Initialized with Live API Integration.
Type 'exit' or 'quit' at any time to end the session.

USER: What is the weather in New York?
ASSISTANT: The live temperature in Newark is 14.6° Celsius.

USER: What is the weather in Rio de Janeiro?
ASSISTANT: The live temperature in Riobamba is 18.7° Celsius.

USER: What is the weather in RiodeJaneiro>
ASSISTANT: Sorry Salman, I couldn't find geographic coordinates for 'RiodeJaneiro>' anywhere on Earth!

USER: exit
ASSISTANT: Goodbye, Salman. Have a wonderful day.
